# TX_CURR Report

Patients Alive and On Treatment at the end of the reporting period, by age group, gender and months of ARVs dispensed (MMD), with All Male / FP / FNP / FBF summary rows. All logic lives in `src/tx_curr.py`.

Set the reporting period, site and report type (**MoH** = 60 defaulter days, **PEPFAR** = 30 defaulter days) in the widgets above, then Run all.

In [0]:
# Report parameters (edit in the widgets bar at the top of the notebook)
dbutils.widgets.text("start_date", "2025-01-01", "1. Start date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2025-03-31", "2. End date (YYYY-MM-DD)")

# Site: searchable dropdown of location names for sites with ART (program 1) data;
# the selected name is mapped back to its location_id (= site_id)
sites = {
    r["name"]: r["location_id"]
    for r in spark.sql("""
        SELECT DISTINCT TRIM(l.name) AS name, l.location_id
        FROM programsdev.malawi.location l
        WHERE l.location_id IN (
            SELECT DISTINCT site_id FROM programsdev.malawi.patient_program WHERE program_id = 1
        )
        ORDER BY name
    """).collect()
}
site_names = list(sites)
default_site = next((n for n, i in sites.items() if i == 865), site_names[0])
dbutils.widgets.dropdown("site_name", default_site, site_names, "3. Site")
dbutils.widgets.dropdown("report_type", "MoH", ["MoH", "PEPFAR"], "4. Report type (MoH 60 days / PEPFAR 30 days)")

start_date = dbutils.widgets.get("start_date").strip()
end_date = dbutils.widgets.get("end_date").strip()
site_name = dbutils.widgets.get("site_name")
site_id = sites[site_name]
report_type = dbutils.widgets.get("report_type")

print(f"{report_type} TX_CURR MMD | period {start_date} to {end_date} | {site_name} (site_id {site_id})")

In [0]:
import os
import sys
import importlib

# Make the repo root (parent of notebooks/) importable so `src` resolves
repo_root = os.path.dirname(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import settings
import src.tx_curr as tx_curr

importlib.reload(tx_curr)  # pick up edits to tx_curr.py without restarting the session

defaulter_days = {
    "MoH": settings.DEFAULTER_DAYS_MOH,       # 60 days
    "PEPFAR": settings.DEFAULTER_DAYS_PEPFAR, # 30 days
}[report_type]

display(tx_curr.tx_curr_by_age_group(start_date, end_date, site_id, defaulter_days=defaulter_days))